# Phase 4 — Player Metrics and Scoring Methodology

This is the authoritative Phase 4 implementation. It reads the corrected Phase 3 structural-pressure dataset, calculates batting and bowling performance separately for IPL and T20I, applies fixed eligibility and scoring rules, and combines format ratings only after format-specific normalization.

The official score uses the complete 2022–2026 population. Recent form is not weighted into the official score. This notebook produces player scores but deliberately does not publish official ranks or top-10 lists; ranking and uncertainty belong to Phase 5.

## Locked methodology

- High pressure is the Phase 3 `High` category; the continuous pressure percentile is used for pressure-gradient metrics.
- Batting records exclude wides, oversteps, and unresolved `other` extras.
- Bowling wicket rate uses delivery records; run suppression uses only no-extras records and is not described as conventional economy.
- Outcome benchmarks remain format-specific and use a declared fallback hierarchy.
- Weights, eligibility thresholds, shrinkage constants, and normalization rules are fixed before player scores are inspected.
- Pressure opportunities determine eligibility and evidence strength; cumulative volume is not a performance bonus.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
INPUT_PATH = ROOT / 'phase3_outputs' / 'phase3_structural_pressure_labelled.csv'
OUTPUT_DIR = ROOT / 'phase4_outputs'
REPORT_PATH = ROOT / 'Phase 4 Player Metrics and Scoring Report.md'
MODEL_VERSION = 'player_scoring_v1'
MIN_BENCHMARK_SUPPORT = 100
FORMATS = ['IPL', 'T20I']

BAT_WEIGHTS = {
    'scoring_quality_z': 0.45,
    'survival_quality_z': 0.25,
    'scoring_pressure_gradient_z': 0.20,
    'survival_pressure_gradient_z': 0.10,
}
BOWL_WEIGHTS = {
    'wicket_quality_z': 0.45,
    'run_suppression_quality_z': 0.30,
    'wicket_pressure_gradient_z': 0.15,
    'run_suppression_pressure_gradient_z': 0.10,
}
BAT_RELIABLE = {'high_records': 80, 'high_matches': 8, 'ordinary_records': 160}
BAT_PROVISIONAL = {'high_records': 40, 'high_matches': 5, 'ordinary_records': 80}
BOWL_RELIABLE = {
    'high_records': 100, 'high_matches': 8, 'ordinary_records': 200,
    'high_clean_records': 80, 'ordinary_clean_records': 160,
}
BOWL_PROVISIONAL = {
    'high_records': 50, 'high_matches': 5, 'ordinary_records': 100,
    'high_clean_records': 40, 'ordinary_clean_records': 80,
}
BAT_SHRINKAGE_K = 80
BOWL_WICKET_SHRINKAGE_K = 100
BOWL_RUN_SHRINKAGE_K = 80
BOWLER_WICKET_TYPES = {'Caught', 'Bowled', 'LBW', 'Stump', 'Hit Wicket'}

assert abs(sum(BAT_WEIGHTS.values()) - 1) < 1e-12
assert abs(sum(BOWL_WEIGHTS.values()) - 1) < 1e-12

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def mode_or_first(values):
    values = values.dropna()
    if values.empty:
        return pd.NA
    modes = values.mode()
    return modes.iloc[0] if not modes.empty else values.iloc[0]


def pressure_slope(frame, x_col, y_col):
    usable = frame[[x_col, y_col]].dropna()
    if len(usable) < 2:
        return np.nan
    x = usable[x_col].astype(float).to_numpy()
    y = usable[y_col].astype(float).to_numpy()
    x_centered = x - x.mean()
    denominator = np.square(x_centered).sum()
    if denominator <= 0:
        return np.nan
    return float((x_centered * (y - y.mean())).sum() / denominator)


def shrink(value, evidence, k):
    if pd.isna(value) or evidence <= 0:
        return np.nan
    return float(value) * float(evidence) / (float(evidence) + float(k))


def attach_context_benchmark(records, outcome_col, metric_name):
    exact_keys = ['format_group', 'innings', 'overid', 'structural_pressure_category']
    stage_keys = ['format_group', 'innings', 'source_over_stage', 'structural_pressure_category']
    broad_keys = ['format_group', 'innings', 'structural_pressure_category']
    levels = [
        ('FORMAT_INNINGS_OVER_PRESSURE', exact_keys),
        ('FORMAT_INNINGS_STAGE_PRESSURE', stage_keys),
        ('FORMAT_INNINGS_PRESSURE', broad_keys),
    ]
    result = records.copy()
    candidate_columns = []
    for index, (level_name, keys) in enumerate(levels):
        stats = (
            records.groupby(keys, observed=True)[outcome_col]
            .agg(['size', 'mean'])
            .reset_index()
            .rename(columns={
                'size': f'_support_{index}',
                'mean': f'_expected_{index}',
            })
        )
        stats[f'_level_{index}'] = level_name
        result = result.merge(stats, on=keys, how='left', validate='many_to_one')
        candidate_columns.extend([f'_support_{index}', f'_expected_{index}', f'_level_{index}'])
    result[f'{metric_name}_benchmark_level'] = pd.NA
    result[f'{metric_name}_benchmark_support'] = np.nan
    result[f'{metric_name}_expected'] = np.nan
    for index in range(len(levels)):
        use = (
            result[f'{metric_name}_benchmark_level'].isna()
            & result[f'_support_{index}'].ge(MIN_BENCHMARK_SUPPORT)
            & result[f'_expected_{index}'].notna()
        )
        result.loc[use, f'{metric_name}_benchmark_level'] = result.loc[use, f'_level_{index}']
        result.loc[use, f'{metric_name}_benchmark_support'] = result.loc[use, f'_support_{index}']
        result.loc[use, f'{metric_name}_expected'] = result.loc[use, f'_expected_{index}']
    require(result[f'{metric_name}_expected'].notna().all(), f'Unbenchmarked {metric_name} records')
    require(result[f'{metric_name}_benchmark_support'].ge(MIN_BENCHMARK_SUPPORT).all(), f'Insufficient {metric_name} support')
    benchmark_table = (
        result[exact_keys + [
            f'{metric_name}_benchmark_level',
            f'{metric_name}_benchmark_support',
            f'{metric_name}_expected',
        ]]
        .drop_duplicates()
        .rename(columns={
            f'{metric_name}_benchmark_level': 'benchmark_level',
            f'{metric_name}_benchmark_support': 'benchmark_support',
            f'{metric_name}_expected': 'expected_outcome',
        })
    )
    benchmark_table.insert(0, 'metric', metric_name)
    return result.drop(columns=candidate_columns), benchmark_table


def classify_batter(row):
    reliable = all(row[key] >= value for key, value in BAT_RELIABLE.items())
    provisional = all(row[key] >= value for key, value in BAT_PROVISIONAL.items())
    return 'Reliable' if reliable else ('Provisional' if provisional else 'Insufficient')


def classify_bowler(row):
    reliable = all(row[key] >= value for key, value in BOWL_RELIABLE.items())
    provisional = all(row[key] >= value for key, value in BOWL_PROVISIONAL.items())
    return 'Reliable' if reliable else ('Provisional' if provisional else 'Insufficient')

## Load and validate the Phase 3 handoff

In [ ]:
deliveries = pd.read_csv(INPUT_PATH, low_memory=False)
deliveries['_row_id'] = np.arange(len(deliveries), dtype=np.int64)
deliveries['is_high_pressure'] = deliveries['structural_pressure_category'].eq('High')
deliveries['is_ordinary_pressure'] = ~deliveries['is_high_pressure']

required_columns = {
    'match_id', 'innings', 'innings_ball_num', 'overid', 'source_over_stage',
    'format_group', 'striker_id', 'striker_name', 'bowler_id', 'bowler_name',
    'runs', 'is_wicket', 'dismissed_player_id', 'dismissal_type_normalized',
    'extras_type', 'structural_pressure_category', 'structural_pressure_percentile',
    'phase3_model_version', 'revised_target', 'first_inn_revised_overs',
    'second_inn_revised_overs',
}
require(required_columns.issubset(deliveries.columns), 'Phase 3 input is missing required columns')
require(not deliveries.duplicated(['match_id', 'innings', 'innings_ball_num']).any(), 'Duplicate Phase 3 keys')
require(set(deliveries['format_group'].unique()) == set(FORMATS), 'Unexpected format population')
require(deliveries['phase3_model_version'].eq('structural_pressure_v1').all(), 'Unexpected Phase 3 model version')
require(not deliveries[['revised_target', 'first_inn_revised_overs', 'second_inn_revised_overs']].notna().any(axis=1).any(), 'Revised match leaked into Phase 4')
require(deliveries['structural_pressure_percentile'].between(0, 1).all(), 'Pressure percentile outside [0,1]')
print(f'Loaded {len(deliveries):,} Phase 3 records across {deliveries.match_id.nunique():,} matches.')

## Build outcome records and format-specific context benchmarks

In [ ]:
batting_mask = deliveries['extras_type'].isna() | deliveries['extras_type'].isin(['byes', 'legbyes'])
batting = deliveries.loc[batting_mask].copy()
batting['batter_dismissal_available'] = ~(batting['is_wicket'] & batting['dismissed_player_id'].isna())
batting['batter_dismissal'] = (
    batting['is_wicket']
    & batting['dismissed_player_id'].notna()
    & batting['dismissed_player_id'].eq(batting['striker_id'])
).astype(int)

bowling_wicket = deliveries.copy()
bowling_wicket['bowler_wicket'] = (
    bowling_wicket['is_wicket']
    & bowling_wicket['dismissal_type_normalized'].isin(BOWLER_WICKET_TYPES)
).astype(int)
bowling_runs = deliveries.loc[deliveries['extras_type'].isna()].copy()

batting_scoring, benchmark_bat_runs = attach_context_benchmark(batting, 'runs', 'batter_runs')
batting_survival, benchmark_bat_dismissal = attach_context_benchmark(
    batting.loc[batting['batter_dismissal_available']].copy(),
    'batter_dismissal',
    'batter_dismissal',
)
bowling_wicket, benchmark_bowler_wicket = attach_context_benchmark(
    bowling_wicket, 'bowler_wicket', 'bowler_wicket'
)
bowling_runs, benchmark_bowler_runs = attach_context_benchmark(
    bowling_runs, 'runs', 'bowler_batter_runs'
)

batting_scoring['scoring_residual'] = batting_scoring['runs'] - batting_scoring['batter_runs_expected']
batting_survival['survival_residual'] = batting_survival['batter_dismissal_expected'] - batting_survival['batter_dismissal']
bowling_wicket['wicket_residual'] = bowling_wicket['bowler_wicket'] - bowling_wicket['bowler_wicket_expected']
bowling_runs['run_suppression_residual'] = bowling_runs['bowler_batter_runs_expected'] - bowling_runs['runs']

outcome_benchmarks = pd.concat([
    benchmark_bat_runs, benchmark_bat_dismissal, benchmark_bowler_wicket, benchmark_bowler_runs
], ignore_index=True)
outcome_benchmarks['phase4_model_version'] = MODEL_VERSION
print(outcome_benchmarks.groupby(['metric', 'benchmark_level'], observed=True).size().to_string())

## Aggregate batting and bowling metrics

In [ ]:
bat_survival_lookup = batting_survival[['_row_id', 'survival_residual']].copy()
bat_events = batting_scoring.merge(bat_survival_lookup, on='_row_id', how='left', validate='one_to_one')

batter_rows = []
for (player_id, format_group), group in bat_events.groupby(['striker_id', 'format_group'], sort=True):
    high = group.loc[group['is_high_pressure']]
    ordinary = group.loc[group['is_ordinary_pressure']]
    high_survival = high['survival_residual'].dropna()
    match_means = high.groupby('match_id', observed=True)['scoring_residual'].mean()
    scoring_quality_raw = 100 * high['scoring_residual'].mean() if len(high) else np.nan
    survival_quality_raw = 100 * high_survival.mean() if len(high_survival) else np.nan
    scoring_gradient_raw = 100 * pressure_slope(group, 'structural_pressure_percentile', 'scoring_residual')
    survival_gradient_raw = 100 * pressure_slope(group, 'structural_pressure_percentile', 'survival_residual')
    batter_rows.append({
        'player_id': int(player_id),
        'player_name': mode_or_first(group['striker_name']),
        'format_group': format_group,
        'high_records': int(len(high)),
        'ordinary_records': int(len(ordinary)),
        'high_matches': int(high['match_id'].nunique()),
        'all_matches': int(group['match_id'].nunique()),
        'high_pressure_runs': int(high['runs'].sum()),
        'ordinary_runs': int(ordinary['runs'].sum()),
        'high_dismissals': int(high['batter_dismissal'].sum()),
        'ordinary_dismissals': int(ordinary['batter_dismissal'].sum()),
        'high_scoring_residual_raw_per100': scoring_quality_raw,
        'high_survival_residual_raw_per100': survival_quality_raw,
        'scoring_pressure_gradient_raw_per100': scoring_gradient_raw,
        'survival_pressure_gradient_raw_per100': survival_gradient_raw,
        'high_match_scoring_residual_sd': match_means.std(ddof=1),
        'high_match_positive_scoring_share': match_means.gt(0).mean() if len(match_means) else np.nan,
    })
batter_metrics = pd.DataFrame(batter_rows)
batter_metrics['eligibility'] = batter_metrics.apply(classify_batter, axis=1)
batter_metrics['scoring_quality'] = batter_metrics.apply(
    lambda row: shrink(row['high_scoring_residual_raw_per100'], row['high_records'], BAT_SHRINKAGE_K), axis=1
)
batter_metrics['survival_quality'] = batter_metrics.apply(
    lambda row: shrink(row['high_survival_residual_raw_per100'], row['high_records'], BAT_SHRINKAGE_K), axis=1
)
batter_metrics['scoring_pressure_gradient'] = batter_metrics.apply(
    lambda row: shrink(row['scoring_pressure_gradient_raw_per100'], row['high_records'], BAT_SHRINKAGE_K), axis=1
)
batter_metrics['survival_pressure_gradient'] = batter_metrics.apply(
    lambda row: shrink(row['survival_pressure_gradient_raw_per100'], row['high_records'], BAT_SHRINKAGE_K), axis=1
)

wicket_rows = []
for (player_id, format_group), group in bowling_wicket.groupby(['bowler_id', 'format_group'], sort=True):
    high = group.loc[group['is_high_pressure']]
    ordinary = group.loc[group['is_ordinary_pressure']]
    match_means = high.groupby('match_id', observed=True)['wicket_residual'].mean()
    wicket_rows.append({
        'player_id': int(player_id),
        'player_name': mode_or_first(group['bowler_name']),
        'format_group': format_group,
        'high_records': int(len(high)),
        'ordinary_records': int(len(ordinary)),
        'high_matches': int(high['match_id'].nunique()),
        'all_matches': int(group['match_id'].nunique()),
        'high_pressure_wickets': int(high['bowler_wicket'].sum()),
        'ordinary_wickets': int(ordinary['bowler_wicket'].sum()),
        'high_wicket_residual_raw_per100': 100 * high['wicket_residual'].mean() if len(high) else np.nan,
        'wicket_pressure_gradient_raw_per100': 100 * pressure_slope(group, 'structural_pressure_percentile', 'wicket_residual'),
        'high_match_wicket_residual_sd': match_means.std(ddof=1),
        'high_match_positive_wicket_share': match_means.gt(0).mean() if len(match_means) else np.nan,
    })
wicket_metrics = pd.DataFrame(wicket_rows)

suppression_rows = []
for (player_id, format_group), group in bowling_runs.groupby(['bowler_id', 'format_group'], sort=True):
    high = group.loc[group['is_high_pressure']]
    ordinary = group.loc[group['is_ordinary_pressure']]
    suppression_rows.append({
        'player_id': int(player_id),
        'format_group': format_group,
        'high_clean_records': int(len(high)),
        'ordinary_clean_records': int(len(ordinary)),
        'high_batter_runs_allowed': int(high['runs'].sum()),
        'ordinary_batter_runs_allowed': int(ordinary['runs'].sum()),
        'high_run_suppression_raw_per100': 100 * high['run_suppression_residual'].mean() if len(high) else np.nan,
        'run_suppression_pressure_gradient_raw_per100': 100 * pressure_slope(group, 'structural_pressure_percentile', 'run_suppression_residual'),
    })
suppression_metrics = pd.DataFrame(suppression_rows)
bowler_metrics = wicket_metrics.merge(
    suppression_metrics, on=['player_id', 'format_group'], how='left', validate='one_to_one'
).fillna({'high_clean_records': 0, 'ordinary_clean_records': 0})
bowler_metrics['eligibility'] = bowler_metrics.apply(classify_bowler, axis=1)
bowler_metrics['wicket_quality'] = bowler_metrics.apply(
    lambda row: shrink(row['high_wicket_residual_raw_per100'], row['high_records'], BOWL_WICKET_SHRINKAGE_K), axis=1
)
bowler_metrics['run_suppression_quality'] = bowler_metrics.apply(
    lambda row: shrink(row['high_run_suppression_raw_per100'], row['high_clean_records'], BOWL_RUN_SHRINKAGE_K), axis=1
)
bowler_metrics['wicket_pressure_gradient'] = bowler_metrics.apply(
    lambda row: shrink(row['wicket_pressure_gradient_raw_per100'], row['high_records'], BOWL_WICKET_SHRINKAGE_K), axis=1
)
bowler_metrics['run_suppression_pressure_gradient'] = bowler_metrics.apply(
    lambda row: shrink(row['run_suppression_pressure_gradient_raw_per100'], row['high_clean_records'], BOWL_RUN_SHRINKAGE_K), axis=1
)

print('Batter eligibility by format')
print(pd.crosstab(batter_metrics['format_group'], batter_metrics['eligibility']).to_string())
print('\nBowler eligibility by format')
print(pd.crosstab(bowler_metrics['format_group'], bowler_metrics['eligibility']).to_string())

## Normalize within format and calculate locked format ratings

In [ ]:
def normalize_and_score(metrics, role, metric_map, weights):
    result = metrics.copy()
    parameters = []
    reliable_mask = result['eligibility'].eq('Reliable')
    for format_group in FORMATS:
        pool_mask = reliable_mask & result['format_group'].eq(format_group)
        require(pool_mask.sum() >= 10, f'Too few reliable {role}s in {format_group}')
        for source_metric, z_column in metric_map.items():
            values = result.loc[pool_mask, source_metric].astype(float)
            require(values.notna().all(), f'Missing reliable {role} metric {source_metric}')
            lower = float(values.quantile(0.025))
            upper = float(values.quantile(0.975))
            winsorized = values.clip(lower=lower, upper=upper)
            mean = float(winsorized.mean())
            sd = float(winsorized.std(ddof=0))
            require(sd > 0, f'Zero normalization SD for {role} {format_group} {source_metric}')
            result.loc[pool_mask, z_column] = (winsorized - mean) / sd
            parameters.append({
                'role': role, 'format_group': format_group, 'metric': source_metric,
                'winsor_lower_2_5pct': lower, 'winsor_upper_97_5pct': upper,
                'winsor_mean': mean, 'winsor_sd': sd, 'reliable_players': int(pool_mask.sum()),
            })
    result['format_composite_z'] = np.nan
    result.loc[reliable_mask, 'format_composite_z'] = sum(
        weight * result.loc[reliable_mask, column] for column, weight in weights.items()
    )
    result['format_rating'] = np.where(
        reliable_mask, 50 + 10 * result['format_composite_z'], np.nan
    )
    result['role'] = role
    result['phase4_model_version'] = MODEL_VERSION
    return result, pd.DataFrame(parameters)


BAT_METRIC_MAP = {
    'scoring_quality': 'scoring_quality_z',
    'survival_quality': 'survival_quality_z',
    'scoring_pressure_gradient': 'scoring_pressure_gradient_z',
    'survival_pressure_gradient': 'survival_pressure_gradient_z',
}
BOWL_METRIC_MAP = {
    'wicket_quality': 'wicket_quality_z',
    'run_suppression_quality': 'run_suppression_quality_z',
    'wicket_pressure_gradient': 'wicket_pressure_gradient_z',
    'run_suppression_pressure_gradient': 'run_suppression_pressure_gradient_z',
}
batter_metrics, bat_normalization = normalize_and_score(
    batter_metrics, 'Batter', BAT_METRIC_MAP, BAT_WEIGHTS
)
bowler_metrics, bowl_normalization = normalize_and_score(
    bowler_metrics, 'Bowler', BOWL_METRIC_MAP, BOWL_WEIGHTS
)
normalization_parameters = pd.concat([bat_normalization, bowl_normalization], ignore_index=True)
normalization_parameters['phase4_model_version'] = MODEL_VERSION

## Combine reliable IPL and T20I ratings

When both formats are reliable, the combined score is their equal mean. When only one format is reliable, the qualifying format score is used; missing-format evidence is never assigned zero.

In [ ]:
def combine_formats(format_metrics, role):
    reliable = format_metrics.loc[format_metrics['eligibility'].eq('Reliable')].copy()
    rows = []
    for player_id, group in reliable.groupby('player_id', sort=True):
        ratings = group.set_index('format_group')['format_rating'].to_dict()
        formats = sorted(ratings)
        combined = float(np.mean([ratings[key] for key in formats]))
        rows.append({
            'player_id': int(player_id),
            'player_name': mode_or_first(group['player_name']),
            'role': role,
            'qualifying_formats': '+'.join(formats),
            'qualifying_format_count': len(formats),
            'ipl_rating': ratings.get('IPL', np.nan),
            't20i_rating': ratings.get('T20I', np.nan),
            'combined_rating': combined,
            'format_combination_rule': 'EQUAL_50_50' if len(formats) == 2 else 'SINGLE_RELIABLE_FORMAT',
            'official_period': 'FULL_2022_2026_NO_RECENCY_WEIGHT',
            'phase4_model_version': MODEL_VERSION,
        })
    return pd.DataFrame(rows).sort_values('player_id', kind='stable').reset_index(drop=True)


batter_combined = combine_formats(batter_metrics, 'Batter')
bowler_combined = combine_formats(bowler_metrics, 'Bowler')
print(f'Official Phase 5 candidate pool: {len(batter_combined)} batters and {len(bowler_combined)} bowlers.')
print('No ranks or top-10 lists are generated in Phase 4.')

## Validation gates

In [ ]:
def validate_combination(combined):
    both = combined['qualifying_format_count'].eq(2)
    one = combined['qualifying_format_count'].eq(1)
    both_ok = np.allclose(
        combined.loc[both, 'combined_rating'],
        combined.loc[both, ['ipl_rating', 't20i_rating']].mean(axis=1),
        rtol=0, atol=1e-8,  # exported ratings are rounded to 8 decimals
    )
    one_expected = combined.loc[one, ['ipl_rating', 't20i_rating']].max(axis=1, skipna=True)
    one_ok = np.allclose(combined.loc[one, 'combined_rating'], one_expected, rtol=0, atol=1e-8)
    return both_ok and one_ok


checks = [
    ('phase3_source_rows', len(deliveries) == 188330, len(deliveries), '188330 corrected Phase 3 records'),
    ('source_key_unique', not deliveries.duplicated(['match_id', 'innings', 'innings_ball_num']).any(), int(deliveries.duplicated(['match_id', 'innings', 'innings_ball_num']).sum()), '0 duplicate delivery keys'),
    ('revised_matches_absent', not deliveries[['revised_target', 'first_inn_revised_overs', 'second_inn_revised_overs']].notna().any(axis=1).any(), 0, '0 revised-schedule records'),
    ('batting_record_rule', not batting['extras_type'].isin(['wide', 'Overstep', 'other']).any(), int(batting['extras_type'].isin(['wide', 'Overstep', 'other']).sum()), '0 disallowed batting records'),
    ('bowling_run_record_rule', bowling_runs['extras_type'].isna().all(), int(bowling_runs['extras_type'].notna().sum()), '0 extras-bearing run-suppression records'),
    ('benchmark_complete', outcome_benchmarks['expected_outcome'].notna().all(), int(outcome_benchmarks['expected_outcome'].isna().sum()), '0 missing expected outcomes'),
    ('benchmark_support', outcome_benchmarks['benchmark_support'].ge(MIN_BENCHMARK_SUPPORT).all(), int(outcome_benchmarks['benchmark_support'].min()), f'minimum support >= {MIN_BENCHMARK_SUPPORT}'),
    ('bat_weights_sum_one', abs(sum(BAT_WEIGHTS.values()) - 1) < 1e-12, sum(BAT_WEIGHTS.values()), 'weights sum to 1'),
    ('bowl_weights_sum_one', abs(sum(BOWL_WEIGHTS.values()) - 1) < 1e-12, sum(BOWL_WEIGHTS.values()), 'weights sum to 1'),
    ('reliable_batter_metrics_complete', batter_metrics.loc[batter_metrics.eligibility.eq('Reliable'), list(BAT_WEIGHTS)].notna().all().all(), int(batter_metrics.loc[batter_metrics.eligibility.eq('Reliable'), list(BAT_WEIGHTS)].isna().sum().sum()), '0 missing reliable batter z-scores'),
    ('reliable_bowler_metrics_complete', bowler_metrics.loc[bowler_metrics.eligibility.eq('Reliable'), list(BOWL_WEIGHTS)].notna().all().all(), int(bowler_metrics.loc[bowler_metrics.eligibility.eq('Reliable'), list(BOWL_WEIGHTS)].isna().sum().sum()), '0 missing reliable bowler z-scores'),
    ('combined_batter_unique', not batter_combined['player_id'].duplicated().any(), int(batter_combined['player_id'].duplicated().sum()), '0 duplicate combined batters'),
    ('combined_bowler_unique', not bowler_combined['player_id'].duplicated().any(), int(bowler_combined['player_id'].duplicated().sum()), '0 duplicate combined bowlers'),
    ('combined_batter_formula', validate_combination(batter_combined), len(batter_combined), 'all combined batter ratings follow format rule'),
    ('combined_bowler_formula', validate_combination(bowler_combined), len(bowler_combined), 'all combined bowler ratings follow format rule'),
    ('no_recent_weight', batter_combined['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all() and bowler_combined['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all(), 'FULL_2022_2026_NO_RECENCY_WEIGHT', 'recent form does not alter official score'),
]
validation = pd.DataFrame(checks, columns=['check_name', 'passed', 'observed', 'acceptance_rule'])
validation['status'] = np.where(validation['passed'], 'PASS', 'FAIL')
validation = validation[['check_name', 'status', 'observed', 'acceptance_rule']]
failures = validation.loc[validation['status'].eq('FAIL')]
require(failures.empty, 'Phase 4 validation failed:\n' + failures.to_string(index=False))
validation

## Export Phase 4 artifacts and report

In [ ]:
eligibility_summary = pd.concat([
    batter_metrics.groupby(['role', 'format_group', 'eligibility'], observed=True).size().rename('player_format_records').reset_index(),
    bowler_metrics.groupby(['role', 'format_group', 'eligibility'], observed=True).size().rename('player_format_records').reset_index(),
], ignore_index=True)
candidate_summary = pd.DataFrame([
    {'role': 'Batter', 'official_candidate_players': len(batter_combined), 'both_format_players': int(batter_combined['qualifying_format_count'].eq(2).sum())},
    {'role': 'Bowler', 'official_candidate_players': len(bowler_combined), 'both_format_players': int(bowler_combined['qualifying_format_count'].eq(2).sum())},
])

methodology = pd.DataFrame([
    ('official_period', 'All eligible 2022-2026 evidence', 'No recent-form weight'),
    ('pressure_quality_sample', 'Phase 3 High category', 'Continuous percentile used for gradients'),
    ('batting_records', 'No extras, byes, legbyes', 'Wide, overstep, other excluded'),
    ('batting_outcomes', 'Batter runs and striker dismissal', 'Dismissal requires dismissed_player_id == striker_id'),
    ('bowling_wicket_records', 'All delivery records', 'Rate is per delivery record, not legal ball'),
    ('bowler_wickets', 'Caught, Bowled, LBW, Stump, Hit Wicket', 'Run outs and non-bowler dismissals excluded'),
    ('bowling_run_records', 'No-extras delivery records', 'Batter-run suppression, not conventional economy'),
    ('benchmark_primary', 'Format x innings x over x pressure category', 'Minimum support 100'),
    ('benchmark_fallback_1', 'Format x innings x stage x pressure category', 'No cross-format pooling'),
    ('benchmark_fallback_2', 'Format x innings x pressure category', 'Observation unavailable if still sparse'),
    ('batting_weights', json.dumps(BAT_WEIGHTS, sort_keys=True), 'Locked before score inspection'),
    ('bowling_weights', json.dumps(BOWL_WEIGHTS, sort_keys=True), 'Locked before score inspection'),
    ('format_combination', '50% IPL + 50% T20I when both reliable', 'Single reliable format used alone; missing is not zero'),
    ('ranking_boundary', 'No official ranks or top 10 in Phase 4', 'Phase 5 owns ranking and uncertainty'),
], columns=['item', 'locked_rule', 'note'])
methodology['phase4_model_version'] = MODEL_VERSION

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = {
    'phase4_batter_format_metrics.csv': batter_metrics.sort_values(['player_id', 'format_group'], kind='stable'),
    'phase4_bowler_format_metrics.csv': bowler_metrics.sort_values(['player_id', 'format_group'], kind='stable'),
    'phase4_batter_combined_scores.csv': batter_combined,
    'phase4_bowler_combined_scores.csv': bowler_combined,
    'phase4_outcome_benchmarks.csv': outcome_benchmarks.sort_values(['metric', 'format_group', 'innings', 'overid', 'structural_pressure_category'], kind='stable'),
    'phase4_normalization_parameters.csv': normalization_parameters,
    'phase4_eligibility_summary.csv': eligibility_summary,
    'phase4_candidate_summary.csv': candidate_summary,
    'phase4_validation_gates.csv': validation,
    'phase4_methodology_specification.csv': methodology,
}
for name, frame in artifacts.items():
    frame.to_csv(OUTPUT_DIR / name, index=False, float_format='%.10g')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_rows = []
for name, frame in artifacts.items():
    path = OUTPUT_DIR / name
    manifest_rows.append({
        'artifact': name, 'rows': len(frame), 'columns': len(frame.columns),
        'bytes': path.stat().st_size, 'sha256': sha256(path), 'model_version': MODEL_VERSION,
    })
manifest = pd.DataFrame(manifest_rows)
phase3_source_sha256 = sha256(INPUT_PATH)
manifest['source_artifact'] = INPUT_PATH.name
manifest['source_sha256'] = phase3_source_sha256
manifest.to_csv(OUTPUT_DIR / 'phase4_output_manifest.csv', index=False)

report = f'''# Phase 4 — Player Metrics and Scoring Report

## Decision

Phase 4 uses all eligible 2022–2026 evidence with no recent-form weighting. IPL and T20I are evaluated and normalized separately, then combined 50/50 only when both formats meet Reliable eligibility. A missing format is never assigned zero.

This refresh reads the regenerated Phase 3 delivery file with SHA-256 `{phase3_source_sha256}`. Its manifest records that source hash for reproducibility.

Phase 4 produces scores and the reliable candidate population. Official ranks and top-10 lists are deferred to Phase 5, where uncertainty and robustness will be assessed.

## Feasible outcomes

Batting uses batter runs and striker-dismissal survival on validated batting records. Bowling uses bowler-credited wickets per delivery record and batter-run suppression on no-extras records. The run-suppression metric is not conventional economy, and no dot-ball metric is claimed.

## Candidate population

- Reliable combined batter candidates: **{len(batter_combined)}**; both-format evidence: **{int(batter_combined.qualifying_format_count.eq(2).sum())}**.
- Reliable combined bowler candidates: **{len(bowler_combined)}**; both-format evidence: **{int(bowler_combined.qualifying_format_count.eq(2).sum())}**.
- Player names and score order are not presented in this Phase 4 report.

## Locked weights

Batting: 45% High-pressure scoring quality, 25% High-pressure survival, 20% scoring-pressure gradient, 10% survival-pressure gradient.

Bowling: 45% High-pressure wicket quality, 30% High-pressure batter-run suppression, 15% wicket-pressure gradient, 10% run-suppression-pressure gradient.

## Validation

All **{len(validation)}** Phase 4 validation gates passed. Benchmark support is at least **{int(outcome_benchmarks.benchmark_support.min())}** records. All reliable metric z-scores and combined ratings are complete.

## Phase 5 handoff

Phase 5 must rank unrounded combined ratings, run fixed and nested match-level bootstrap analysis, test leave-one-player-out outcome benchmarks, evaluate weight/eligibility/pressure-definition sensitivity, and inspect the rank 8–15 boundary before publishing the official top 10.
'''
REPORT_PATH.write_text(report, encoding='utf-8')
print(validation.to_string(index=False))
print('\nOutput manifest')
print(manifest.to_string(index=False))

## Persisted-output verification

This cell rereads the exported tables and manifest. It verifies hashes, uniqueness, score completeness, and the format-combination rule after persistence.

In [ ]:
persisted_manifest = pd.read_csv(OUTPUT_DIR / 'phase4_output_manifest.csv')
persisted_bat = pd.read_csv(OUTPUT_DIR / 'phase4_batter_combined_scores.csv')
persisted_bowl = pd.read_csv(OUTPUT_DIR / 'phase4_bowler_combined_scores.csv')
hashes_match = all(
    sha256(OUTPUT_DIR / row.artifact) == row.sha256
    for row in persisted_manifest.itertuples(index=False)
)
persisted_checks = pd.DataFrame([
    ('manifest_hashes_match', hashes_match),
    ('persisted_batters_unique', not persisted_bat['player_id'].duplicated().any()),
    ('persisted_bowlers_unique', not persisted_bowl['player_id'].duplicated().any()),
    ('persisted_batter_scores_complete', persisted_bat['combined_rating'].notna().all()),
    ('persisted_bowler_scores_complete', persisted_bowl['combined_rating'].notna().all()),
    ('persisted_batter_combination_valid', validate_combination(persisted_bat)),
    ('persisted_bowler_combination_valid', validate_combination(persisted_bowl)),
    ('persisted_no_recent_weight', persisted_bat['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all() and persisted_bowl['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all()),
], columns=['check_name', 'passed'])
persisted_checks['status'] = np.where(persisted_checks['passed'], 'PASS', 'FAIL')
require(persisted_checks['passed'].all(), 'Persisted Phase 4 verification failed')
persisted_checks[['check_name', 'status']]